# 06 - Generación de resultados por caso

Este notebook genera una carpeta `analysis_outputs` dentro de cada caso.

Así cada repositorio queda con sus propias tablas, gráficas y resumen.

In [9]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PYCEFR_TABLE = OUTPUT_DIR / "pycefr_constructs_global.csv"
RADON_TABLE = OUTPUT_DIR / "radon_functions_global.csv"
CROSS_TABLE = OUTPUT_DIR / "04_radon_pycefr_cross_global.csv"
INTERESTING_TABLE = OUTPUT_DIR / "05_casos_interesantes_global.csv"
CATALOG_TABLE = OUTPUT_DIR / "catalog_global.csv"


def read_table(path):
    if Path(path).exists():
        return pd.read_csv(path)
    print(f"No encontrado: {path}")
    return pd.DataFrame()


df_pycefr_all = read_table(PYCEFR_TABLE)
df_radon_all = read_table(RADON_TABLE)
df_cross_all = read_table(CROSS_TABLE)
df_interesting_all = read_table(INTERESTING_TABLE)
catalog = read_table(CATALOG_TABLE)

# tables we have so far:
print("PyCEFR:", df_pycefr_all.shape)
print("Radon:", df_radon_all.shape)
print("Cruce:", df_cross_all.shape)
print("Casos interesantes:", df_interesting_all.shape)
print("Catálogo:", catalog.shape)

PyCEFR: (35647, 11)
Radon: (1961, 15)
Cruce: (1961, 17)
Casos interesantes: (1961, 18)
Catálogo: (2, 3)


In [11]:
catalog, df_pycefr_all, df_radon_all = load_all_processed()
print(df_pycefr_all.shape, df_radon_all.shape)

(35647, 11) (1961, 15)


In [12]:
print("PyCEFR por proyecto:")
display(
    df_pycefr_all
    .groupby("case")
    .size()
    .reset_index(name="n_constructs")
    .sort_values("n_constructs", ascending=False)
)

print("Radon por proyecto:")
display(
    df_radon_all
    .groupby("case")
    .size()
    .reset_index(name="n_radon_items")
    .sort_values("n_radon_items", ascending=False)
)

PyCEFR por proyecto:


,case,n_constructs
2,python-algorithms,21724
4,python-mini-projects,10907
0,30-Days-Of-Python,1358
3,python-beginner-programming-exercises,1176
1,Analisis-CC-PyCEFR,482


Radon por proyecto:


,case,n_radon_items
2,python-algorithms,1474
4,python-mini-projects,340
3,python-beginner-programming-exercises,109
0,30-Days-Of-Python,22
1,Analisis-CC-PyCEFR,16


In [13]:
summary = (
    df_pycefr_all
    .groupby("case")
    .size()
    .reset_index(name="pycefr_constructs")
    .merge(
        df_radon_all
        .groupby("case")
        .size()
        .reset_index(name="radon_items"),
        on="case",
        how="outer"
    )
    .fillna(0)
)

summary["pycefr_radon_ratio"] = (
    summary["pycefr_constructs"] / summary["radon_items"].replace(0, pd.NA)
).round(2)

summary = summary.sort_values("pycefr_constructs", ascending=False)

display(summary)

,case,pycefr_constructs,radon_items,pycefr_radon_ratio
2,python-algorithms,21724,1474,14.74
4,python-mini-projects,10907,340,32.08
0,30-Days-Of-Python,1358,22,61.73
3,python-beginner-programming-exercises,1176,109,10.79
1,Analisis-CC-PyCEFR,482,16,30.12


In [14]:
for case in sorted(set(catalog["case"])):
    case_dir = PROCESSED_DIR / case
    out = case_dir / "analysis_outputs"
    out.mkdir(exist_ok=True)

    py = df_pycefr_all[df_pycefr_all["case"] == case].copy()
    ra = df_radon_all[df_radon_all["case"] == case].copy()
    cr = cross_radon_pycefr(ra, py)
    it = label_interesting_cases(cr)

    py.to_csv(out / "pycefr_constructs.csv", index=False)
    ra.to_csv(out / "radon_functions.csv", index=False)
    cr.to_csv(out / "radon_pycefr_cross.csv", index=False)
    it.to_csv(out / "interesting_cases.csv", index=False)

    summary = {
        "case": case,
        "n_pycefr_constructs": int(len(py)),
        "n_pycefr_classes": int(py["class"].nunique()) if not py.empty else 0,
        "max_pycefr_level": LEVEL_ORDER_INV.get(int(py["level_num"].max())) if not py.empty and pd.notna(py["level_num"].max()) else None,
        "n_radon_functions": int(len(ra)),
        "max_radon_complexity": float(ra["complexity"].max()) if not ra.empty else None,
        "max_radon_grade": RADON_GRADE_ORDER_INV.get(int(ra["radon_grade_num"].max())) if not ra.empty and pd.notna(ra["radon_grade_num"].max()) else None,
        "n_interesting_cases": int((it["case_type"] != "Sin patrón destacado").sum()) if not it.empty else 0,
    }
    with open(out / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    if not py.empty:
        level_counts = py["level"].value_counts().reindex(LEVEL_ORDER.keys(), fill_value=0)
        plt.figure(figsize=(7,4))
        level_counts.plot.bar()
        plt.title(f"Niveles PyCEFR - {case}")
        plt.xlabel("Nivel")
        plt.ylabel("Constructos")
        plt.tight_layout()
        plt.savefig(out / "pycefr_level_distribution.png", dpi=150)
        plt.close()

        top_classes = py["class"].value_counts().head(15)
        plt.figure(figsize=(8,5))
        top_classes.sort_values().plot.barh()
        plt.title(f"Constructos PyCEFR más frecuentes - {case}")
        plt.tight_layout()
        plt.savefig(out / "pycefr_top_classes.png", dpi=150)
        plt.close()

    if not ra.empty:
        grade_counts = ra["radon_grade"].value_counts().reindex(RADON_GRADE_ORDER.keys(), fill_value=0)
        plt.figure(figsize=(7,4))
        grade_counts.plot.bar()
        plt.title(f"Calificaciones de Radon - {case}")
        plt.xlabel("Calificación")
        plt.ylabel("Funciones")
        plt.tight_layout()
        plt.savefig(out / "radon_calificaciones.png", dpi=150)
        plt.close()

        plt.figure(figsize=(7,4))
        ra["complexity"].dropna().plot.hist(bins=15)
        plt.title(f"Complejidad ciclomática - {case}")
        plt.xlabel("Complejidad")
        plt.tight_layout()
        plt.savefig(out / "radon_complexity_histogram.png", dpi=150)
        plt.close()

    if not cr.empty:
        plt.figure(figsize=(7,5))
        plt.scatter(cr["complexity"], cr["max_level_num"], alpha=0.7)
        plt.title(f"Radon vs PyCEFR - {case}")
        plt.xlabel("Complejidad Radon")
        plt.ylabel("Nivel máximo PyCEFR")
        plt.yticks(list(LEVEL_ORDER_INV.keys()), list(LEVEL_ORDER_INV.values()))
        plt.tight_layout()
        plt.savefig(out / "radon_vs_pycefr_scatter.png", dpi=150)
        plt.close()

    print("Generado:", out)

Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/30-Days-Of-Python/analysis_outputs
Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/Analisis-CC-PyCEFR/analysis_outputs
Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/python-algorithms/analysis_outputs
Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/python-beginner-programming-exercises/analysis_outputs
Generado: /home/juan/Documents/Analisis-CC-PyCEFR/data/processed/python-mini-projects/analysis_outputs
